In [ ]:
##########################
### views.py - CloudBW ###
##########################

from django.shortcuts import render, redirect
from django.http import HttpResponse
from django.views.decorators.csrf import csrf_exempt
from project.jwt_tooling import decode_jwt
from ics import Calendar, Event
import requests
import json
import datetime

static = "/var/www/static"

### fremder Code - Funktion jwt_tooling ###

def jwt_login(request):
    token = request.GET.get("token")
    if not token:
        return HttpResponse("Kein Token übergeben.", status=400)

    try:
        daten = decode_jwt(token)
    except Exception:
        return HttpResponse("Ungültiges oder abgelaufenes Token.", status=401)

    buerger_id = daten.get("user_id")
    if not buerger_id:
        return HttpResponse("Token enthält keine buerger_id.", status=400)

    # Session auf Server B setzen
    request.session["user_id"] = buerger_id

    # Weiter ins Dashboard
    return redirect("start") #hier anpassen, weiterleiten auf die Zielseite

### fremder Code - Ende ###

def start(request):
	buerger_id = request.session.get("user_id")

	if not buerger_id:
		return render(request, "app/start.html")
	else:
		response = requests.get(f"http://[2001:7c0:2320:2:f816:3eff:feb6:6731]:8000/api/buerger/beruf/{buerger_id}")
		beruf = response.json()["beruf"]

		request.session["beruf"] = beruf
		return render(request, "app/start.html")

def terminspezialisierung(request):
	with open(f"{static}/arztregister.json", "r") as datei:
		arztregister = json.load(datei)

	# Spezialisierungen und Standorte aus Arztregister auslesen und jeder Spezialisierung id und standort das arztes zuweisen
	spezialisierungen = []
	for arzt in list(arztregister["personen"]):
		spezialisierung = arzt["spezialisierung"]
		
		if spezialisierung not in spezialisierungen:
			spezialisierungen.append(spezialisierung)
	
	context = {"spezialisierungen": spezialisierungen}

	return render(request, "app/terminspezialisierung.html", context)
	
def terminstandort(request):
	with open(f"{static}/arztregister.json", "r") as datei:
		arztregister = json.load(datei)	

	# Spezialisierung abfangen
	if request.method == 'POST':
		spezialisierung = request.POST.get("spezialisierung")

		# Standorte auslesen die auf die gewählte Spezialisierung passen
		standorte = []
		for arzt in list(arztregister["personen"]):
			if arzt["spezialisierung"] == spezialisierung and arzt["standort"] not in standorte:
				standorte.append(arzt["standort"])
	
		context = {
			"spezialisierung": spezialisierung,
			"standorte": standorte
		}
			
		return render(request, "app/terminstandort.html", context)

def terminarzt(request):
	with open(f"{static}/arztregister.json", "r") as datei:
		arztregister = json.load(datei)

	if request.method == 'POST':
		spezialisierung = request.POST.get("spezialisierung")
		standort = request.POST.get("standort")

		# Ärzte auslesen die auf Spezialisierung und Standort passen
		aerzte = []
		for arzt in list(arztregister["personen"]):
			if arzt["spezialisierung"] == spezialisierung and arzt["standort"] == standort:
				response = requests.get(f'http://[2001:7c0:2320:2:f816:3eff:fef8:f5b9]:8000/einwohnermeldeamt/api/person/{arzt["uid"]}', headers={"Connection": "close"})
				personendaten = response.json()
				vorname = personendaten["vorname"]
				if personendaten["nachname_neu"]:
					nachname = personendaten["nachname_neu"]
				else:
					nachname = personendaten["nachname_geburt"]
				arztname = f"Dr. {vorname} {nachname}"
				aerzte.append({"uid": arzt["uid"], "name": arztname})

		context = {
			"spezialisierung": spezialisierung,
			"standort": standort,
			"aerzte": aerzte
		}

		return render(request, "app/terminarzt.html", context)

def termindatum(request):
	if request.method == 'POST':

		spezialisierung = request.POST.get("spezialisierung")
		standort = request.POST.get("standort")
		arzt = request.POST.get("arzt")

		context = {
			"spezialisierung": spezialisierung,
			"standort": standort,
			"arzt": arzt
		}

		return render(request, "app/termindatum.html", context)

def terminzeit(request):
	if request.method == 'POST':

		spezialisierung = request.POST.get("spezialisierung")
		standort = request.POST.get("standort")
		arzt = request.POST.get("arzt")
		datum = request.POST.get("datum")

		context = {
			"spezialisierung": spezialisierung,
			"standort": standort,
			"arzt": arzt,
			"datum": datum
		}

		return render(request, "app/terminzeit.html", context)
	
def terminendetest(request):
	if request.method == 'POST':

		spezialisierung = request.POST.get("spezialisierung")
		standort = request.POST.get("standort")
		arztuid = request.POST.get("arzt")
		datum = request.POST.get("datum")
		uhrzeit = request.POST.get("uhrzeit")

		response = requests.get(f'http://[2001:7c0:2320:2:f816:3eff:fef8:f5b9]:8000/einwohnermeldeamt/api/person/{arztuid}', headers={"Connection": "close"})
		personendaten = response.json()
		vorname = personendaten["vorname"]
		if personendaten["nachname_neu"]:
			nachname = personendaten["nachname_neu"]
		else:
			nachname = personendaten["nachname_geburt"]
		arztname = f"Dr. {vorname} {nachname}"

		startzeit = datetime.datetime.strptime(f"{datum} {uhrzeit}", "%Y-%m-%d %H:%M")
		endzeit = startzeit + datetime.timedelta(minutes=30)

		event = Event()
		event.name = f"Arzttermin: {arztname}, {spezialisierung}, {standort}"
		event.begin = startzeit
		event.end = endzeit
		event.location = standort
		event.description = f"Arzttermin bei:\n{arztname}, {spezialisierung}\nin folgendem Krankenhaus:\n{standort}.\nWir wünschen gute Besserung!"
		
		calendar = Calendar()
		calendar.events.add(event)

		response = HttpResponse(calendar.serialize(), content_type="text/calendar")
		response["Content-Disposition"] = 'attachment; filename="termin.ics"'
		return response

def videoabgabe(request):
	return render(request, "app/videoabgabe.html")

In [ ]:
############################
### start.html - CloudBW ###
############################

{% load static %}
<!DOCTYPE html>
<html lang="de">
<head>
        <meta charset="utf-8"></meta>
        <meta name="viewport" content="width=device-width,initial-scale=1"></meta>
        <title>Gesundheit - Startseite</title>
        <link rel="stylesheet" type="text/css" href="{% static 'styles.css' %}">
</head>
<body>
        <h1>Willkommen in der Abteilung Gesundheit!</h1>

        {% if not request.session.user_id %}
        <form action="http://[2001:7c0:2320:2:f816:3eff:fef8:f5b9]:8000/einwohnermeldeamt/login">
        <button type="submit">Zum Login</button>
        </form>
        {% endif %}

        {% if request.session.beruf == "arzt" %}
        <h2>Sie sind Arzt?</h2>
        <div class="link-container">
            <form action="http://raspberrypi.local:8000/geburt/">
            <button type="submit">Geburt anzeigen</button>
            </form>
            <form action="http://raspberrypi.local:8000/tod/">
            <button type="submit">Sterbefall anzeigen</button>
            </form>
        </div>
        {% endif %}

        {% if request.session.user_id %}
        <h2>Sie sind Bürger?</h2>
        <div class="link-container">
            <form action="http://[2001:7c0:2320:2:f816:3eff:fe06:8d56]:8000/terminspezialisierung/">
            <button type="submit">Terminvereinbarungssystem</button>
            </form>
        </div>
        {% endif %}

</body>

In [ ]:
###########################################
### terminspezialisierung.html -CloudBW ###
###########################################

{% load static %}
<!DOCTYPE html>
<html lang="de">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <link rel="stylesheet" type="text/css" href="{% static 'styles.css' %}">
    <title>Arzttermin vereinbaren</title>
    <style>
        body {
            font-family: Arial, sans-serif;
            max-width: 600px;
            margin: 0 auto;
            padding: 20px;
        }
    </style>
</head>
<body>
    <h1>Arzttermin vereinbaren</h1>
    <form action="{% url 'terminstandort' %}" method="POST" id="arzttermin">
    {% csrf_token %}

    <!-- Spezialisierung -->
    <label for="spezialisierung">Spezialisierung:</label>
    <select name="spezialisierung" id="spezialisierung" onchange="submit()">
        <option value="" selected>Bitte wählen</option>
        {% for spezial in spezialisierungen %}
            <option value="{{ spezial }}">{{ spezial }}</option>
        {% endfor %}
    </select>

    <!-- Standort - hier noch leer!-->
    <label for="standort">Standort:</label>
    <select name="standort" id="standort" disabled>
        <option value="" disabled selected>Bitte zuerst Spezialisierung wählen!</option>
    </select>

    <!-- Arzt - hier noch leer! -->
    <label for="arzt">Arzt:</label>
    <select name="arzt" id="arzt" disabled>
        <option value="" disabled selected>Bitte zuerst Spezialisierung und Standort wählen</option>
    </select>
</form>
</body>
</html>

In [ ]:
#####################################
### terminstandort.html - CloudBW ###
#####################################

{% load static %}
<!DOCTYPE html>
<html lang="de">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <link rel="stylesheet" type="text/css" href="{% static 'styles.css' %}">
    <title>Arzttermin vereinbaren</title>
    <style>
        body {
            font-family: Arial, sans-serif;
            max-width: 600px;
            margin: 0 auto;
            padding: 20px;
        }
    </style>
</head>
<body>
    <h1>Personendaten erfassen</h1>
    <form action="{% url 'terminarzt' %}" method="POST" id="arzttermin">
    {% csrf_token %}

    <!-- Spezialisierung -->
    <label for="spezialisierung">Spezialisierung:</label>
    <select name="spezialisierung" id="spezialisierung" onchange="submit()">
        <option value="{{ spezialisierung }}" selected>{{ spezialisierung }}</option>
    </select>

    <!-- Standort - hier noch leer!-->
    <label for="standort">Standort:</label>
    <select name="standort" id="standort" onchange="submit()">
        <option value="" selected>Bitte wählen</option>
        {% for ort in standorte %}
            <option value="{{ ort }}">{{ ort }}</option>
        {% endfor %}
    </select>

    <!-- Arzt - hier noch leer! -->
    <label for="arzt">Arzt:</label>
    <select name="arzt" id="arzt" disabled>
        <option value="" disabled selected>Bitte zuerst Spezialisierung und Standort wählen</option>
    </select>
</form>
</body>
</html>

In [ ]:
#################################
### terminarzt.html - CloudBW ###
#################################

{% load static %}
<!DOCTYPE html>
<html lang="de">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <link rel="stylesheet" type="text/css" href="{% static 'styles.css' %}">
    <title>Arzttermin vereinbaren</title>
    <style>
        body {
            font-family: Arial, sans-serif;
            max-width: 600px;
            margin: 0 auto;
            padding: 20px;
        }
    </style>
</head>
<body>
    <h1>Personendaten erfassen</h1>
    <form action="{% url 'termindatum' %}" method="POST" id="arzttermin">
    {% csrf_token %}

    <!-- Spezialisierung -->
    <label for="spezialisierung">Spezialisierung:</label>
    <select name="spezialisierung" id="spezialisierung" onchange="submit()">
        <option value="{{ spezialisierung }}" selected>{{ spezialisierung }}</option>
    </select>

    <!-- Standort - hier noch leer!-->
    <label for="standort">Standort:</label>
    <select name="standort" id="standort" onchange="submit()">
        <option value="{{ standort }}" selected>{{ standort }}</option>
    </select>

    <!-- Arzt - hier noch leer! -->
    <label for="arzt">Arzt:</label>
     <select name="arzt" id="arzt">
        <option value="" selected>Bitte wählen</option>
        {% for arzt in aerzte %}
            <option value="{{ arzt.uid }}">{{ arzt.name }}</option>
        {% endfor %}
    </select>

    <button type="submit">Termin buchen!</button>
</form>
</body>
</html>

In [ ]:
################################
### termindatum.html CloudBW ###
################################

{% load static %}
<!DOCTYPE html>
<html lang="de">
<head>
    <meta charset="UTF-8">
    <title>Termin auswählen</title>

    <link rel="stylesheet" href="https://cdn.jsdelivr.net/npm/flatpickr/dist/flatpickr.min.css">
    <link rel="stylesheet" type="text/css" href="{% static 'styles.css' %}">

    <style>
        body {
            font-family: Arial, sans-serif;
        }

        .container {
            display: flex;
            gap: 40px;
            margin-top: 40px;
        }

        .left, .right {
            width: 50%;
        }

        label {
            display: block;
            margin-bottom: 10px;
            font-weight: bold;
        }
    </style>
</head>
<body>

<h1>Terminbuchung</h1>

<form method="POST" action="{% url 'terminzeit' %}">
    {% csrf_token %}

    <div class="container">

        <!-- Linke Seite: Kalender -->
        <div class="left">
            <label for="datum">Datum auswählen</label>
            <input type="hidden" name ="datum" id="datum" onchange="submit()">

            <!-- Flatpickr JS - streng genommen nicht mein Code -->
            <script src="https://cdn.jsdelivr.net/npm/flatpickr"></script>

            <script>
                flatpickr("#datum", {
                    inline: true,
                    dateFormat: "Y-m-d",
                    locale: "de"
                });
            </script>
            <!-- streng genommen nicht mein Code - Ende -->

        </div>

        <!-- Rechte Seite: Uhrzeiten -->
        <div class="right">
                <label for="uhrzeit">Uhrzeit auswählen</label>
                <select id="uhrzeit" name="uhrzeit" disabled>
                    <option value="">Bitte wählen</option>
                    <option value="09:00">09:00</option>
                    <option value="10:00">10:00</option>
                    <option value="11:00">11:00</option>
                    <option value="13:00">13:00</option>
                    <option value="14:00">14:00</option>
                    <option value="15:00">15:00</option>
                </select>
                <p>Bitte zuerst ein Datum auswählen.</p>

        </div>

        <input type="hidden" name="spezialisierung" value="{{ spezialisierung }}">
        <input type="hidden" name="standort" value="{{ standort }}">
        <input type="hidden" name="arzt" value="{{ arzt }}">

    </div>

</form>

</body>
</html>

In [ ]:
#################################
### terminzeit-html - CloudBW ###
#################################

{% load static %}
<!DOCTYPE html>
<html lang="de">
<head>
    <meta charset="UTF-8">
    <title>Termin auswählen</title>

    <link rel="stylesheet" href="https://cdn.jsdelivr.net/npm/flatpickr/dist/flatpickr.min.css">
    <link rel="stylesheet" type="text/css" href="{% static 'styles.css' %}">

    <style>
        body {
            font-family: Arial, sans-serif;
        }

        .container {
            display: flex;
            gap: 40px;
            margin-top: 40px;
        }

        .left, .right {
            width: 50%;
        }

        label {
            display: block;
            margin-bottom: 10px;
            font-weight: bold;
        }
    </style>
</head>
<body>

<h1>Terminbuchung</h1>

<form method="POST" action="{% url 'terminendetest' %}">
    {% csrf_token %}

    <div class="container">

        <!-- Linke Seite: Kalender -->
        <div class="left">
            <label for="datum">Datum auswählen</label>
            <input value="{{ datum }}" id="datum" name="datum" onchange="submit()">

            <!-- Flatpickr JS -->
            <script src="https://cdn.jsdelivr.net/npm/flatpickr"></script>

            <script>
                flatpickr("#datum", {
                    inline: true,
                    dateFormat: "Y-m-d",
                    locale: "de"
                });
            </script>
        </div>


        <!-- Rechte Seite: Uhrzeiten -->
        <div class="right">
                <label for="uhrzeit">Uhrzeit auswählen</label>
                <select id="uhrzeit" name="uhrzeit">
                    <option value="">Bitte wählen</option>
                    <option value="09:00">09:00</option>
                    <option value="10:00">10:00</option>
                    <option value="11:00">11:00</option>
                    <option value="13:00">13:00</option>
                    <option value="14:00">14:00</option>
                    <option value="15:00">15:00</option>
                </select>

                <input type="hidden" name="spezialisierung" value="{{ spezialisierung }}">
                <input type="hidden" name="standort" value="{{ standort }}">
                <input type="hidden" name="arzt" value="{{ arzt }}">
                <input type="hidden" name="datum" value="{{ datum }}">

                <button type="submit">Abschicken.</button>
        </div>

    </div>

</form>

</body>
</html>

In [ ]:
##################################
### videoabgabe.html - CloudBW ###
##################################

{% load static %}
<!DOCTYPE html>
<html lang="de">
<head>
        <meta charset="utf-8"></meta>
        <meta name="viewport" content="width=device-width,initial-scale=1"></meta>
        <title>Videoabgabe</title>
        <style>
        body {
            margin: 0;
            height: 100vh;
            display: flex;
            justify-content: center;
            align-items: center;
            background-color: #000;
        }

        video {
            max-width: 78%;
            height: auto;
        }
        </style>
</head>
<body>
    <video controls>
    <source src="{% static 'videoabgabe.mp4' %}" type="video/mp4">
    Dein Browser unterstützt dieses Video nicht.
    </video>
</body>

In [ ]:
#################################
### styles.css - Raspberry Pi ###
#################################

body {
    font-family: Arial, sans-serif;
    max-width: 800px;
    margin: 0 auto;
    padding: 20px;
    }

h1 {
    font-size: 2.5em;
    color: #45a049;
    margin-bottom: 30px;
    }

h2 {
    font-size: 1.8em;
    color: #45a049;
    margin-top: 20px;
    margin-bottom: 15px;
    }   

/* 
.arztformular {
    margin-bottom: 15px;
    } */

label {
    display: block;
    margin-bottom: 5px;
    font-weight: bold;
    }

input[type="text"],
input[type="date"] {
    width: 100%;
    padding: 8px;
    border: 1px solid #d1d1d1;
    border-radius: 4px;
    }

button {
    background-color: #4CAF50;
    color: white;
    padding: 10px 15px;
    border: none;
    border-radius: 4px;
    cursor: pointer;
    margin-bottom: 10px;
    }

button:hover {
    background-color: #45a049;
    }

In [ ]:
##############################################
### arztregisteraktualisieren.py - CloudBW ###
##############################################

import requests
import json
import random
import datetime

static = "/var/www/static"

spezialisierungen = ["Neurologe", "Pulmologe", "Kardiologe", "Zahnarzt", "Dentalchirurg", "Unfallchirurg", "Allgemeinmediziner", "Gynäkologe", "Psychiater"]

def arztregistrieren():

    response = requests.get("http://[2001:7c0:2320:2:f816:3eff:feb6:6731]:8000/api/personenliste/arzt", headers={"Connection": "close"})
    arztliste = response.json()
    print(arztliste)

    with open(f"{static}/arztregister.json", "r", encoding="utf-8") as datei:
        arztregister = json.load(datei)
        print(arztregister)

    #Prüfen ob alle Ärzte aus dem Register auch wirklich noch Ärzte sind. Falls nicht entfernen.
    for arzt in list(arztregister["personen"]):
        if arzt["uid"] not in arztliste.values():
            arztregister["personen"].remove(arzt)   
        else:
            continue #Platzhalter
        print(arztregister)

    #Arztliste auslesen und alle Ärzte dem Register hinzufügen
    for arzt in arztliste["personen"]:
        buerger_id = arzt["uid"]
        print(buerger_id)

        if buerger_id in list(arztregister["personen"]):
            continue

        #Arzt aus der API-Liste unserem Arzt-Register hinzufügen - wenn er noch nicht registriert ist
        elif buerger_id not in list(arztregister["personen"]):
            spezialisierung = random.choice(spezialisierungen)
            standort = f"{arzt['arbeitgeber']}, {arzt['adresse']}"
            arztregister["personen"].append({"uid": buerger_id, "spezialisierung": spezialisierung, "standort": standort})

        else:
            continue #Platzhalter
    
    print(arztregister)

arztregistrieren()

In [ ]:
######################################
### schichteinteilung.py - CloudBW ###
######################################

import json
import datetime

static = "/var/www/static"

schichten = ["früh", "spät", "nacht", "frei"]

def schichteneinteilen():

    with open(f"{static}/arztregister.json", "r", encoding="utf-8") as datei:
        arztregister = json.load(datei)
        print(arztregister)

    # Listen mit Standorten anlegen als Grundlage für die Schichtverteilung
    standorte = dict()
    for arzt in arztregister["personen"]:
        standort = arzt["standort"]
        if standort not in standorte:
            standorte[standort] = list()
        standorte[standort].append(arzt)

    for standort, aerzte in standorte.items():

        schichtzaehler = {"früh": 0, "spät": 0, "nacht": 0, "frei": 0}
        neueAerzte = list()

        for arzt in aerzte:
            # bereits eingetragene Ärzte haben auch schon eine Schicht
            if "schicht" in arzt:
                schichtzaehler[arzt["schicht"]] += 1
            # neue Ärzte aber nicht
            else:
                neueAerzte.append(arzt)
        
        # neue Ärzte der Schicht mit den wenigsten Ärzten hinzufügen
        for arzt in neueAerzte:
            kleinsteschicht = schichten[0]
            kleinsteanzahl = schichtzaehler[kleinsteschicht]

            for schicht in schichten:
                if schichtzaehler[schicht] <= kleinsteanzahl:
                    kleinsteschicht = schicht
                    kleinsteanzahl = schichtzaehler[schicht]

            arzt["schicht"] = kleinsteschicht
            schichtzaehler[kleinsteschicht] += 1

        #Alle Ärzte einmal rotieren
        for arzt in aerzte:
            if "schicht" in arzt:
                letzteSchicht = arzt["schicht"]
                letzterIndex = schichten.index(letzteSchicht)
                if letzterIndex == 3:
                    neuerIndex = 0
                else:
                    neuerIndex = letzterIndex +1
                neueSchicht = schichten[neuerIndex]
                arzt["schicht"] = neueSchicht
                schichtzaehler[neueSchicht] += 1

    with open(f"{static}/arztregister.json", "w", encoding="utf-8") as datei:
        json.dump(arztregister, datei, indent=4)

    with open(f"{static}/aktualisierung.txt", "w") as datei:
        datei.write(f"Arztregister aktualisiert am {datetime.datetime.now()}.")

schichteneinteilen()

In [ ]:
###################################
### arztregister.json - CloudBW ###
###################################

{
    "personen": [
        {
            "uid": "4c3c74f9-4b33-4321-8e49-d9b1cadadf23",
            "spezialisierung": "Neurologe",
            "standort": "Katharinenhospital, Kriegsbergstra\u00dfe 60, 70174 Stuttgart",
            "schicht": "frei"
        },
        {
            "uid": "3848720f-91f1-4b3c-bf71-c1d8c504dacc",
            "spezialisierung": "Neurologe",
            "standort": "Katharinenhospital, Kriegsbergstra\u00dfe 60, 70174 Stuttgart",
            "schicht": "nacht"
        },
        {
            "uid": "4d26b3f3-f3be-42b0-9029-b13b736d0386",
            "spezialisierung": "Pulmologe",
            "standort": "RKH Klinikum, Posilipostra\u00dfe 4, 71640 Ludwigsburg",
            "schicht": "frei"
        },
        {
            "uid": "8e3a914c-d999-46cf-aa8b-a36c2272b9c7",
            "spezialisierung": "Neurologe",
            "standort": "Katharinenhospital, Kriegsbergstra\u00dfe 60, 70174 Stuttgart",
            "schicht": "sp\u00e4t"
        },
        {
            "uid": "0835c6d1-7202-4d19-a83e-e38f45050c22",
            "spezialisierung": "Kardiologe",
            "standort": "RKH Klinikum, Posilipostra\u00dfe 4, 71640 Ludwigsburg",
            "schicht": "nacht"
        },
        {
            "uid": "867dc473-9499-4732-8585-1cfc6c4f2d04",
            "spezialisierung": "Pulmologe",
            "standort": "RKH Klinikum, Posilipostra\u00dfe 4, 71640 Ludwigsburg",
            "schicht": "sp\u00e4t"
        },
        {
            "uid": "00681072-83a3-402f-90b7-5d69dd8aac3e",
            "spezialisierung": "Unfallchirurg",
            "standort": "Katharinenhospital, Kriegsbergstra\u00dfe 60, 70174 Stuttgart",
            "schicht": "fr\u00fch"
        },
        {
            "uid": "4273626a-6e34-4e17-b873-2250acbc1ddd",
            "spezialisierung": "Kardiologe",
            "standort": "RKH Klinikum, Posilipostra\u00dfe 4, 71640 Ludwigsburg",
            "schicht": "fr\u00fch"
        }
    ]
}

In [ ]:
####################################
### aktualisierung.txt - CloudBW ###
####################################

Arztregister aktualisiert am 2026-02-26 20:16:59.413083.

In [ ]:
#########################
### crontab - CloudBW ###
#########################

0 2 * * * /usr/bin/python3 /var/www/static/arztregisteraktualisieren.py

0 2 * * 1 /usr/bin/python3 /var/www/static/schichteinteilung.py

In [ ]:
###############################
### views.py - Raspberry Pi ###
###############################

from django.shortcuts import render, redirect
from django.http import HttpResponse
import RPi.GPIO as GPIO
from mfrc522 import SimpleMFRC522
import requests
import time
from project.jwt_tooling import decode_jwt
#from fpdf import FPDF

def jwt_login(request):
    token = request.GET.get("token")
    if not token:
        return HttpResponse("Kein Token übergeben.", status=400)

    try:
        daten = decode_jwt(token)
        print(f"Raspberry daten: {daten}")
        print(f"Raspberry user-id: {daten['user_id']}")
    except Exception:
        return HttpResponse("Ungültiges oder abgelaufenes Token.", status=401)

    buerger_id = daten.get("user_id")
    if not buerger_id:
        return HttpResponse("Token enthält keine buerger_id.", status=400)

    # Session auf Server B setzen
    request.session["user_id"] = buerger_id

    # Weiter ins Dashboard
    return redirect("geburt") #hier anpassen, weiterleiten auf die Zielseite

def geburt(request):
	if request.method == 'POST':

		if "scan_mutter" in request.POST:
			reader = SimpleMFRC522()
			id, data = reader.read()
			id_mutter = data.strip()
			print(id_mutter)
			request.session["id_mutter"] = id_mutter
			return redirect("geburt")

		if "scan_vater" in request.POST:

			reader = SimpleMFRC522()
			id, data = reader.read()
			id_vater = data.strip()
			print(id_vater)
			request.session["id_vater"] = id_vater
			return redirect("geburt")

		if "geburt" in request.POST:

			nachname = request.POST.get("nachname_geburt")
			vorname = request.POST.get("vorname")
			geburtsdatum = request.POST.get("geburtsdatum")
			id_mutter = request.session["id_mutter"]
			id_vater = request.session["id_vater"]

			person = {"nachname_geburt": nachname, "vorname": vorname, "geburtsdatum": geburtsdatum, "vater_id": id_vater, "mutter_id": id_mutter}
			elterngeld = {"id_vater": id_vater, "id_mutter": id_mutter}
			print(elterngeld)

			neugeboren = requests.post("http://[2001:7c0:2320:2:f816:3eff:fef8:f5b9]:8000/einwohnermeldeamt/personenstandsregister_api", data=person, headers={"Connection": "close"})
			print(f"Personenregister: {neugeboren}")

			buerger_id = neugeboren.text
			id_kind = {"id_kind": buerger_id}

			kindergeld = requests.post("http://[2001:7c0:2320:2:f816:3eff:fed4:e456]:1810/kindergeldanlegen", data=id_kind, headers={"Connection": "close"})
			print(f"Kindergeld: {kindergeld}")

			elterngeldanlegen = requests.post("http://[2001:7c0:2320:2:f816:3eff:fed4:e456]:1810/elterngeldanlegen", data=elterngeld, headers={"Connection": "close"})
			print(f"Elterngeld: {elterngeldanlegen}")

			# Zurückgegebene Bürger ID auf RFID Karte schreiben
			reader = SimpleMFRC522()
			reader.write(buerger_id)
			print(f"RFID-Karte beschrieben mit {buerger_id}.")

			# IDs wieder aus der Session löschen 
			del request.session["id_mutter"]
			del request.session["id_vater"]

			return HttpResponse(f"""
                       			<html>
                       				<head>
                       					<title>Herzlichen Glückwunsch!</title>
									</head>
                       					<body style="display:flex; justify-content:center; align-items:center">
                       						<div style="text-align:center">
                       							<h1 style="font-size:3rem; color:#45a049">
                       								Herzlichen Glückwunsch zur Geburt von {vorname}.
												</h1>
											</div>
										</body>
								</html>
								""")
	else:
		return render(request, "app/geburt.html")

def tod(request):
	if request.method == 'POST':

		sterbedatum = request.POST.get("sterbedatum")

		# Bürger ID aus RFID Karte auslesen
		reader = SimpleMFRC522()
		id, data = reader.read()
		# Keine Ahnung wo sich ein unsichtbares Zeichen dazugemogelt hat - aber mit strip() gehts!
		id_person = data.strip()
		print(id_person)

		response = requests.get(f"http://[2001:7c0:2320:2:f816:3eff:fef8:f5b9]:8000/einwohnermeldeamt/api/person/{id_person}", headers={"Connection": "close"})
		print(response)
		personendaten = response.json()
		print(personendaten)
		vorname = personendaten["vorname"]
		if personendaten["nachname_neu"]:
			nachname = personendaten["nachname_neu"]
		else:
			nachname = personendaten["nachname_geburt"]
		name_person = f"{vorname} {nachname}"

		person = {"buerger_id": id_person, "sterbedatum": sterbedatum}

		response = requests.post("http://[2001:7c0:2320:2:f816:3eff:fef8:f5b9]:8000/einwohnermeldeamt/personenstandsregister_api", data=person, headers={"Connection": "close"})
		print(response)

		return HttpResponse(f"""
                       			<html>
                       				<head>
                       					<title>Herzlichen Glückwunsch!</title>
									</head>
                       					<body style="display:flex; justify-content:center; align-items:center">
                       						<div style="text-align:center">
                       							<h1 style="font-size:3rem; color:#45a049">
                       								{name_person} ist am {sterbedatum} verstorben. Requiescat in Pace.
												</h1>
											</div>
										</body>
								</html>
							""")

	else:
		return render(request, "app/tod.html")

In [ ]:
##################################
### geburt.html - Raspberry Pi ###
##################################

{% load static %}
<!DOCTYPE html>
<html lang="de">

<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <link rel="stylesheet" type="text/css" href="{% static 'styles.css' %}">
    <title>Geburt erfassen</title>
</head>

<body>
    <h1>Personendaten erfassen</h1>
    <form method="post">
        {% csrf_token %}

        {% if not request.session.id_mutter %}
            <!-- Diese Button wird eingeblendet, wenn keine "id_mutter" in der Session gespeichert ist -->
            <p>Bitte scannen Sie zuerst die RFID-Karte der Mutter:</p>
            <button type="submit" name="scan_mutter">RFID-Karte der Mutter scannen</button>

        {% elif not request.session.id_vater %}
            <!-- Wenn eine "id_mutter" in der Session gespeichert ist, wird dieser Button eingeblendet -->
            <p>Bitte scannen Sie als nächstes die RFID-Karte des Vaters:</p>
            <button type="submit" name="scan_vater">RFID-Karte des Vaters scannen</button>

        {% else %}
            <!-- Eine Formular für die Geburt wird erst dann eingeblendet, wenn sowohl "id_mutter", als auch "id_vater" in der Session stehen! -->
            <div class="arztformular">
                <label for="nachname">Nachname:</label>
                <input type="text" id="nachname_geburt" name="nachname_geburt" required>
            </div>

            <div class="arztformular">
                <label for="vorname">Vorname:</label>
                <input type="text" id="vorname" name="vorname" required>
            </div>

            <div class="arztformular">
                <label for="geburtsdatum">Geburtsdatum:</label>
                <input type="date" id="geburtsdatum" name="geburtsdatum" required>
            </div>

            <button type="submit" name="geburt">Geburt registrieren</button>
        {% endif %}
    </form>
</body>
</html>


In [ ]:
###############################
### tod.html - Raspberry Pi ###
###############################

{% load static %}
<!DOCTYPE html>
<html lang="de">

<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <link rel="stylesheet" type="text/css" href="{% static 'styles.css' %}">
    <title>Tod erfassen</title>
</head>

<body>
    <h1>Personendaten erfassen</h1>
    <form method="post">
        {% csrf_token %}

        <div class="arztformular">
            <label for="sterbedatum">Sterbedatum:</label>
            <input type="date" id="sterbedatum" name="sterbedatum" required>
        </div>

        <button type="submit">Tod bestätigen.</button>
    </form>
</body>
</html>

In [ ]:
#################################
### styles.css - Raspberry Pi ###
#################################

body {
    font-family: Arial, sans-serif;
    max-width: 800px;
    margin: 0 auto;
    padding: 20px;
    }

.arztformular {
    margin-bottom: 15px;
    }

label {
    display: block;
    margin-bottom: 5px;
    font-weight: bold;
    }

input[type="text"],
input[type="date"] {
    width: 100%;
    padding: 8px;
    border: 1px solid #d1d1d1;
    border-radius: 4px;
    }

button {
    background-color: #4CAF50;
    color: white;
    padding: 10px 15px;
    border: none;
    border-radius: 4px;
    cursor: pointer;
    }

button:hover {
    background-color: #45a049;
    }